In [7]:
import guppylang
guppylang.enable_experimental_features()
from guppylang import guppy, qubit
from guppylang.std.builtins import dagger, power, result, control
from guppylang.std.qsystem import measure
from guppylang.std.quantum import angle, h, x, rx, s
from guppylang.std.modifier import NormalizeGuppy, apply_passes, build_emu

from hugr.hugr.render import RenderConfig

@guppy(unitary=True)
def bar(q: qubit) -> None:
    x(q)
    # for _ in range(1):

@guppy
def foo() -> None:
    q1 = qubit()
    q2 = qubit()
    h(q1)
    with control(q1):
        bar(q2) #, angle(1/3))
    # with dagger, control(q1):
    #     rx(q2, angle(1/3))
        # h(q2) #
    
    # measure(q1)
    # measure(q2)

    result("q1", measure(q1))
    result("q2", measure(q2))

hugr = foo.compile().modules[0]

# hugr = NormalizeGuppy(simplify_cfgs=False, 
#                       remove_tuple_untuple=False,
#                       remove_dead_funcs=False,
#                       constant_folding=False,
#                       inline_dfgs=False,
#                       remove_redundant_order_edges=True,
#                       squash_borrows=False,
#                     )(hugr)

hugr = NormalizeGuppy()(hugr)
# hugr.render_dot(RenderConfig(max_node_label_length=1000, display_node_id=True)).view(filename="before")
hugr = apply_passes(hugr)
hugr.render_dot(RenderConfig(max_node_label_length=1000, display_node_id=True)).view(filename="after")


build_emu(foo, 3, False).with_shots(1000).run().collated_counts()

Counter({(('q1', '0'), ('q2', '1')): 519, (('q1', '1'), ('q2', '1')): 481})

## Control

In [12]:
import guppylang
guppylang.enable_experimental_features()
from guppylang import guppy, qubit
from guppylang.std.builtins import dagger, power, result, control
from guppylang.std.qsystem import measure
from guppylang.std.quantum import h, x, s
from guppylang.std.modifier import apply_passes


@guppy
def foo() -> None:
    q1 = qubit()
    q2 = qubit()
    
    h(q1) 
    with control(q1):
        x(q2)
        h(q2)
        s(q2)

    
    # measure(q1)

    result("q1", measure(q1))
    result("q2", measure(q2))

hugr = foo.compile().modules[0]



hugr = apply_passes(hugr)
hugr.render_dot().view()

'__main__.gv.pdf'

In [13]:
from guppylang.std.modifier import build_emu

build_emu(foo, 2).with_shots(50).run().collated_counts()


Counter({(('q1', '0'), ('q2', '0')): 23,
         (('q1', '1'), ('q2', '0')): 16,
         (('q1', '1'), ('q2', '1')): 11})


### Semantic tests: verifying `control` modifier correctness

Each test runs the emulator and asserts known quantum-mechanical outcomes.

| Test group | What it checks |
|---|---|
| ctrl=\|0⟩ → no-op | body is **never** executed when ctrl is \|0⟩ |
| ctrl=\|1⟩ → fires | gates with observable bit-flips always execute |
| Bell state | H + controlled-X creates entanglement → correlated outcomes |
| Toffoli (2 ctrls) | target flips iff **both** controls are \|1⟩ |
| Multi-gate block | sequences / cancellations inside one `with control` |
| Function calls | `@guppy(control=True)` helpers are lifted correctly |
| Loops (comptime) | classical loops unrolled at compile time under control |
| Branching (comptime) | compile-time if/else selects the right gate under control |


#### Setup: extra imports and assertion helpers

In [14]:
from guppylang.std.quantum import y, z, t, tdg, s, sdg, rx, ry, rz
from guppylang.std.angles  import angle, pi

def run(func, n_qubits, shots=200, norm=True):
    """Compile and emulate `func`, return collated counts."""
    return build_emu(func, n_qubits, norm).with_shots(shots).run().collated_counts()

def assert_deterministic(counts, **expected):
    """Assert that every shot produced the single expected outcome dict."""
    assert len(counts) == 1, \
        f"Expected a single outcome, got multiple: {dict(counts)}"
    actual = dict(next(iter(counts)))
    for tag, val in expected.items():
        assert actual.get(tag) == val, \
            f"Tag '{tag}': expected '{val}', got '{actual.get(tag)}' in {actual}"

def assert_outcomes_seen(counts, *outcomes):
    """Assert each given outcome dict appears at least once in the counts."""
    seen = [dict(k) for k in counts.keys()]
    for outcome in outcomes:
        assert any(
            all(s.get(t) == v for t, v in outcome.items()) for s in seen
        ), f"Expected outcome {outcome} not seen. Got: {seen}"

def assert_correlated(counts, tag_a, tag_b, same_value=True):
    """Assert tag_a and tag_b always take the same (or opposite) values."""
    for key in counts:
        d = dict(key)
        if same_value:
            assert d.get(tag_a) == d.get(tag_b), \
                f"Expected {tag_a}=={tag_b}, got {d}"
        else:
            assert d.get(tag_a) != d.get(tag_b), \
                f"Expected {tag_a}!={tag_b}, got {d}"

print("Helpers ready.")


Helpers ready.


#### ctrl=|0⟩: body is a complete no-op

When the control qubit is |0⟩ the entire block is skipped regardless of which gates are inside. We run several gates that would produce a detectable bit-flip and confirm both qubits still measure as |0⟩.

In [15]:
@guppy
def ctrl0_x() -> None:
    ctrl   = qubit()     # |0⟩
    target = qubit()     # |0⟩
    with control(ctrl):
        x(target)        # would flip, but ctrl=|0⟩
    result("ctrl",   measure(ctrl))
    result("target", measure(target))

@guppy
def ctrl0_y() -> None:
    ctrl   = qubit()
    target = qubit()
    with control(ctrl):
        y(target)
    result("ctrl",   measure(ctrl))
    result("target", measure(target))

@guppy
def ctrl0_rx_pi() -> None:
    ctrl   = qubit()
    target = qubit()
    with control(ctrl):
        rx(target, pi)
    result("ctrl",   measure(ctrl))
    result("target", measure(target))

@guppy
def ctrl0_ry_pi() -> None:
    ctrl   = qubit()
    target = qubit()
    with control(ctrl):
        ry(target, pi)
    result("ctrl",   measure(ctrl))
    result("target", measure(target))

@guppy
def ctrl0_h() -> None:
    ctrl   = qubit()
    target = qubit()
    with control(ctrl):
        h(target)        # H|0⟩ = |+⟩, but only when ctrl=|1⟩
    result("ctrl",   measure(ctrl))
    result("target", measure(target))

@guppy
def ctrl0_s() -> None:
    ctrl   = qubit()
    target = qubit()
    with control(ctrl):
        s(target)
    result("ctrl",   measure(ctrl))
    result("target", measure(target))

@guppy
def ctrl0_t() -> None:
    ctrl   = qubit()
    target = qubit()
    with control(ctrl):
        t(target)
    result("ctrl",   measure(ctrl))
    result("target", measure(target))

for func in [ctrl0_x, ctrl0_y, ctrl0_rx_pi, ctrl0_ry_pi, ctrl0_h, ctrl0_s, ctrl0_t]:
    counts = run(func, 2)
    assert_deterministic(counts, ctrl="0", target="0")
    print(f"✓ {func.wrapped.name}: ctrl=|0⟩ → body skipped, both qubits remain |0⟩")


✓ ctrl0_x: ctrl=|0⟩ → body skipped, both qubits remain |0⟩
✓ ctrl0_y: ctrl=|0⟩ → body skipped, both qubits remain |0⟩
✓ ctrl0_rx_pi: ctrl=|0⟩ → body skipped, both qubits remain |0⟩
✓ ctrl0_ry_pi: ctrl=|0⟩ → body skipped, both qubits remain |0⟩
✓ ctrl0_h: ctrl=|0⟩ → body skipped, both qubits remain |0⟩
✓ ctrl0_s: ctrl=|0⟩ → body skipped, both qubits remain |0⟩
✓ ctrl0_t: ctrl=|0⟩ → body skipped, both qubits remain |0⟩


#### ctrl=|1⟩: gates with observable bit-flip effects

With the control qubit forced to |1⟩, every gate inside the block executes. We test all gates whose action on |0⟩ is measurably |1⟩.

In [16]:
@guppy
def ctrl1_x() -> None:
    """Controlled-X (CNOT): X|0⟩ = |1⟩ → target flips."""
    ctrl   = qubit(); x(ctrl)
    target = qubit()
    with control(ctrl):
        x(target)
    result("ctrl",   measure(ctrl))
    result("target", measure(target))

@guppy
def ctrl1_y() -> None:
    """Controlled-Y: Y|0⟩ = i|1⟩ → target measures as |1⟩."""
    ctrl   = qubit(); x(ctrl)
    target = qubit()
    with control(ctrl):
        y(target)
    result("ctrl",   measure(ctrl))
    result("target", measure(target))

@guppy
def ctrl1_rx_pi() -> None:
    """Controlled-Rx(π): Rx(π)|0⟩ = -i|1⟩ → target measures as |1⟩."""
    ctrl   = qubit(); x(ctrl)
    target = qubit()
    with control(ctrl):
        rx(target, pi)
    result("ctrl",   measure(ctrl))
    result("target", measure(target))

@guppy
def ctrl1_ry_pi() -> None:
    """Controlled-Ry(π): Ry(π)|0⟩ = |1⟩ → target measures as |1⟩."""
    ctrl   = qubit(); x(ctrl)
    target = qubit()
    with control(ctrl):
        ry(target, pi)
    result("ctrl",   measure(ctrl))
    result("target", measure(target))

for func in [ctrl1_x, ctrl1_y, ctrl1_rx_pi, ctrl1_ry_pi]:
    counts = run(func, 2)
    assert_deterministic(counts, ctrl="1", target="1")
    print(f"✓ {func.wrapped.name}: ctrl=|1⟩ → gate fires, target flips to |1⟩")

# apply_passes(ctrl1_ry_pi.compile().modules[0]).render_dot().view()


✓ ctrl1_x: ctrl=|1⟩ → gate fires, target flips to |1⟩
✓ ctrl1_y: ctrl=|1⟩ → gate fires, target flips to |1⟩
✓ ctrl1_rx_pi: ctrl=|1⟩ → gate fires, target flips to |1⟩
✓ ctrl1_ry_pi: ctrl=|1⟩ → gate fires, target flips to |1⟩


#### Bell-state test

H(ctrl) creates a superposition. Applying controlled-X entangles both qubits into the Bell state (|00⟩ + |11⟩)/√2. Measurement outcomes must always be identical (both 0 or both 1).

In [17]:
@guppy
def bell_state() -> None:
    ctrl   = qubit()
    h(ctrl)              # |+⟩ = (|0⟩ + |1⟩)/√2
    target = qubit()     # |0⟩
    with control(ctrl):
        x(target)        # CX: creates (|00⟩ + |11⟩)/√2
    result("ctrl",   measure(ctrl))
    result("target", measure(target))

counts_bell = run(bell_state, 2, shots=500)
assert_correlated(counts_bell, "ctrl", "target", same_value=True)
assert_outcomes_seen(counts_bell,
    {"ctrl": "0", "target": "0"},
    {"ctrl": "1", "target": "1"})
print(f"✓ bell_state: ctrl and target always agree; counts = {dict(counts_bell)}")


✓ bell_state: ctrl and target always agree; counts = {(('ctrl', '1'), ('target', '1')): 244, (('ctrl', '0'), ('target', '0')): 256}


#### Multiple control qubits: Toffoli gate

`with control(ctrl1, ctrl2): x(target)` is semantically a Toffoli gate. The target flips only when **both** controls are |1⟩.

In [18]:
@guppy
def toffoli_11() -> None:
    """ctrl1=|1⟩, ctrl2=|1⟩ → target flips to |1⟩."""
    ctrl1  = qubit(); x(ctrl1)
    ctrl2  = qubit(); x(ctrl2)
    target = qubit()
    with control(ctrl1, ctrl2):
        x(target)
    result("ctrl1",  measure(ctrl1))
    result("ctrl2",  measure(ctrl2))
    result("target", measure(target))

@guppy
def toffoli_10() -> None:
    """ctrl1=|1⟩, ctrl2=|0⟩ → target stays |0⟩."""
    ctrl1  = qubit(); x(ctrl1)
    ctrl2  = qubit()
    target = qubit()
    with control(ctrl1, ctrl2):
        x(target)
    result("ctrl1",  measure(ctrl1))
    result("ctrl2",  measure(ctrl2))
    result("target", measure(target))

@guppy
def toffoli_01() -> None:
    """ctrl1=|0⟩, ctrl2=|1⟩ → target stays |0⟩."""
    ctrl1  = qubit()
    ctrl2  = qubit(); x(ctrl2)
    target = qubit()
    with control(ctrl1, ctrl2):
        x(target)
    result("ctrl1",  measure(ctrl1))
    result("ctrl2",  measure(ctrl2))
    result("target", measure(target))

counts_11 = run(toffoli_11, 3)
assert_deterministic(counts_11, ctrl1="1", ctrl2="1", target="1")
print("✓ toffoli_11: ctrl1=ctrl2=|1⟩  → target flips")

counts_10 = run(toffoli_10, 3)
assert_deterministic(counts_10, ctrl1="1", ctrl2="0", target="0")
print("✓ toffoli_10: ctrl1=|1⟩ ctrl2=|0⟩ → target unchanged")

counts_01 = run(toffoli_01, 3)
assert_deterministic(counts_01, ctrl1="0", ctrl2="1", target="0")
print("✓ toffoli_01: ctrl1=|0⟩ ctrl2=|1⟩ → target unchanged")


✓ toffoli_11: ctrl1=ctrl2=|1⟩  → target flips
✓ toffoli_10: ctrl1=|1⟩ ctrl2=|0⟩ → target unchanged
✓ toffoli_01: ctrl1=|0⟩ ctrl2=|1⟩ → target unchanged


#### Multi-gate blocks: sequences and cancellations

All gates inside a single `with control(ctrl):` block are wrapped together. Pairs that cancel each other (e.g. X²=I) leave the target unchanged even when ctrl=|1⟩. We also test with superposition control states to exercise entangled outcomes.

In [19]:
@guppy
def double_x_cancel() -> None:
    """ctrl=|1⟩, controlled X·X = I → target stays |0⟩."""
    ctrl   = qubit(); x(ctrl)
    target = qubit()
    with control(ctrl):
        x(target)
        x(target)          # second X cancels first: X² = I
    result("ctrl",   measure(ctrl))
    result("target", measure(target))

@guppy
def x_s_sdg() -> None:
    """ctrl=|1⟩, controlled X·S·S† = X (phase terms cancel) → target=|1⟩."""
    ctrl   = qubit(); x(ctrl)
    target = qubit()
    with control(ctrl):
        x(target)          # |0⟩ → |1⟩
        s(target)          # S|1⟩ = i|1⟩
        sdg(target)        # S†|i|1⟩ = |1⟩ (S·S† = I on the phase)
    result("ctrl",   measure(ctrl))
    result("target", measure(target))

@guppy
def x_h_superpos() -> None:
    """ctrl=|1⟩, controlled X·H·X creates superposition → both target values."""
    ctrl   = qubit(); x(ctrl)
    target = qubit()
    with control(ctrl):
        x(target)          # |0⟩ → |1⟩
        h(target)          # H|1⟩ = |−⟩
        x(target)          # X|−⟩ = -|−⟩ (still 50/50 on measurement)
    result("ctrl",   measure(ctrl))
    result("target", measure(target))

counts_dbl  = run(double_x_cancel, 2)
assert_deterministic(counts_dbl, ctrl="1", target="0")
print("✓ double_x_cancel: ctrl=|1⟩, X·X = I  → target stays |0⟩")
print(f"counts: {dict(counts_dbl)}")

counts_ssdg = run(x_s_sdg, 2)
assert_deterministic(counts_ssdg, ctrl="1", target="1")
print("✓ x_s_sdg:         ctrl=|1⟩, X·S·S†   → target measures as |1⟩")
print(f"counts: {dict(counts_ssdg)}")

counts_xhx  = run(x_h_superpos, 2, shots=500)
assert_outcomes_seen(counts_xhx,
    {"ctrl": "1", "target": "0"},
    {"ctrl": "1", "target": "1"})
print("✓ x_h_superpos:    ctrl=|1⟩, X·H·X    → superposition (both target values seen)")
print(f"counts: {dict(counts_xhx)}")

# Superposition control state
@guppy
def x_s_sdg_bell() -> None:
    """ctrl=|+⟩, controlled X·S·S† = X → Bell state (|00⟩ + |11⟩)/√2."""
    ctrl   = qubit(); h(ctrl)
    target = qubit()
    with control(ctrl):
        x(target)
        s(target)
        sdg(target)
    result("ctrl",   measure(ctrl))
    result("target", measure(target))

@guppy
def x_h_superpos_bell() -> None:
    """ctrl=Rx(π/3)|0⟩, controlled X·H·X → mixed entangled output."""
    ctrl   = qubit(); rx(ctrl, angle(1/3))
    target = qubit()
    with control(ctrl):
        x(target)
        h(target)
        x(target)
    result("ctrl",   measure(ctrl))
    result("target", measure(target))

counts_ssdg = run(x_s_sdg_bell, 2)
print("✓ x_s_sdg_bell:         ctrl=|+⟩, X·S·S† → |00⟩ + |11⟩")
print(f"counts: {dict(counts_ssdg)}")

counts_xhx  = run(x_h_superpos_bell, 2, shots=1000)
print("✓ x_h_superpos_bell:    ctrl=rx(pi/3)|0⟩, controlled X·H·X -> 75% |00⟩ 25% |1-⟩")
print(f"counts: {dict(counts_xhx)}")


✓ double_x_cancel: ctrl=|1⟩, X·X = I  → target stays |0⟩
counts: {(('ctrl', '1'), ('target', '0')): 200}
✓ x_s_sdg:         ctrl=|1⟩, X·S·S†   → target measures as |1⟩
counts: {(('ctrl', '1'), ('target', '1')): 200}
✓ x_h_superpos:    ctrl=|1⟩, X·H·X    → superposition (both target values seen)
counts: {(('ctrl', '1'), ('target', '1')): 269, (('ctrl', '1'), ('target', '0')): 231}
✓ x_s_sdg_bell:         ctrl=|+⟩, X·S·S† → |00⟩ + |11⟩
counts: {(('ctrl', '1'), ('target', '1')): 101, (('ctrl', '0'), ('target', '0')): 99}
✓ x_h_superpos_bell:    ctrl=rx(pi/3)|0⟩, controlled X·H·X -> 75% |00⟩ 25% |1-⟩
counts: {(('ctrl', '0'), ('target', '0')): 763, (('ctrl', '1'), ('target', '1')): 117, (('ctrl', '1'), ('target', '0')): 120}



#### Complex programs inside `control`: function calls, loops, and branching


##### Loop standard guppy

In [20]:

# ctrl=|1⟩ tests
@guppy
def test_loop_odd_x() -> None:
    """ctrl=|1⟩, controlled-X³ (loop) = X → target flips to |1⟩."""
    ctrl   = qubit(); x(ctrl)
    target = qubit()
    with control(ctrl):
        for _ in range(3):
            x(target)
    result("ctrl",   measure(ctrl))
    result("target", measure(target))

@guppy
def test_loop_even_x() -> None:
    """ctrl=|1⟩, controlled-X⁴ (loop) = I → target stays |0⟩."""
    ctrl   = qubit(); x(q=ctrl)
    target = qubit()
    with control(ctrl):
        for _ in range(4):
            x(target)
    result("ctrl",   measure(ctrl))
    result("target", measure(target))

# ctrl=|0⟩ tests — body must be skipped regardless of loop count
@guppy
def test_loop_ctrl0_odd_x() -> None:
    """ctrl=|0⟩, controlled-X³ → body skipped, target stays |0⟩."""
    ctrl   = qubit()           # |0⟩
    target = qubit()
    with control(ctrl):
        for _ in range(3):
            x(target)
    result("ctrl",   measure(ctrl))
    result("target", measure(target))



from hugr.hugr.render import RenderConfig

# test_loop_odd_x.compile().modules[0].render_dot(RenderConfig(display_node_id=True, max_node_label_length=1000)).view()
counts_odd  = run(test_loop_odd_x,  2, norm=False)
assert_deterministic(counts_odd,  ctrl="1", target="1")
print("✓ test_loop_odd_x:        ctrl=|1⟩, comptime-unrolled X³ → target flips to |1⟩")
counts_even = run(test_loop_even_x, 2)
assert_deterministic(counts_even, ctrl="1", target="0")
print("✓ test_loop_even_x:       ctrl=|1⟩, comptime-unrolled X⁴ = I → target stays |0⟩")

counts_c0_odd  = run(test_loop_ctrl0_odd_x,  2)
# assert_deterministic(counts_c0_odd,  ctrl="0", target="0")
print("✓ test_loop_ctrl0_odd_x:  ctrl=|0⟩, loop body skipped (X³) → target stays |0⟩")
print(f"counts: {dict(counts_c0_odd)}")



PyModifierResolverError: Modifier Node(32) applied to the node CFG with more than one node found. cannot be modified

##### Branching in normal guppy

In [21]:
@guppy(control=True)
def conditional_gate(q: qubit, use_x: bool) -> None:
    """Apply X (deterministic flip) or Rx(π/2) (creates superposition)."""
    if use_x:
        x(q)
    else:
        rx(q, pi / 2)  # partial rotation → 50/50 measurement

# ctrl=|1⟩ tests
@guppy
def test_branch_x() -> None:
    """ctrl=|1⟩, comptime selects X branch → target deterministically flips."""
    ctrl   = qubit(); x(ctrl)
    target = qubit()
    with control(ctrl):
        conditional_gate(target, True)
    result("ctrl",   measure(ctrl))
    result("target", measure(target))

@guppy
def test_branch_rx() -> None:
    """ctrl=|1⟩, comptime selects Rx(π/2) branch → target in superposition."""
    ctrl   = qubit(); x(ctrl)
    target = qubit()
    with control(ctrl):
        conditional_gate(target, False)
    result("ctrl",   measure(ctrl))
    result("target", measure(target))

# ctrl=|0⟩ tests — body must be skipped regardless of selected branch
@guppy
def test_branch_ctrl0_x() -> None:
    """ctrl=|0⟩, comptime selects X branch → body skipped, target stays |0⟩."""
    ctrl   = qubit()           # |0⟩
    target = qubit()
    with control(ctrl):
        conditional_gate(target, True)
    result("ctrl",   measure(ctrl))
    result("target", measure(target))

@guppy
def test_branch_ctrl0_rx() -> None:
    """ctrl=|0⟩, comptime selects Rx(π/2) branch → body skipped, target stays |0⟩."""
    ctrl   = qubit()           # |0⟩
    target = qubit()
    with control(ctrl):
        conditional_gate(target, False)
    result("ctrl",   measure(ctrl))
    result("target", measure(target))

counts_bx  = run(test_branch_x,  2)
assert_deterministic(counts_bx, ctrl="1", target="1")
print("✓ test_branch_x:        ctrl=|1⟩, comptime→X branch  → target flips to |1⟩")

counts_brx = run(test_branch_rx, 2, shots=500)
assert_outcomes_seen(counts_brx,
    {"ctrl": "1", "target": "0"},
    {"ctrl": "1", "target": "1"})
print("✓ test_branch_rx:       ctrl=|1⟩, comptime→Rx branch → target in superposition (both outcomes seen)")

counts_c0_x  = run(test_branch_ctrl0_x,  2)
assert_deterministic(counts_c0_x,  ctrl="0", target="0")
print("✓ test_branch_ctrl0_x:  ctrl=|0⟩, comptime→X branch  → body skipped, target stays |0⟩")

counts_c0_rx = run(test_branch_ctrl0_rx, 2)
assert_deterministic(counts_c0_rx, ctrl="0", target="0")
print("✓ test_branch_ctrl0_rx: ctrl=|0⟩, comptime→Rx branch → body skipped, target stays |0⟩")


PyModifierResolverError: Modifier Node(29) applied to the node CFG with more than one node found. cannot be modified

##### Function call inside `control`

A `@guppy(control=True)` helper function can be called inside a `with control` block. The control modifier is transparently lifted over the callee.

In [ ]:
@guppy(control=True)
def apply_x_twice(q: qubit) -> None:
    """Apply X twice: net effect is identity (X² = I)."""
    x(q)
    x(q)

@guppy(control=True)
def apply_x_then_z(q: qubit) -> None:
    """Apply X then Z.  X flips |0⟩→|1⟩; Z adds only a phase → still |1⟩."""
    x(q)
    z(q)

@guppy(control=True)
def apply_h_then_x(q: qubit) -> None:
    """Apply H then X.  H|0⟩=|+⟩, X|+⟩=|−⟩ → superposition, both outcomes seen."""
    h(q)
    x(q)

@guppy
def test_func_call_identity() -> None:
    """ctrl=|1⟩, controlled helper (X²=I) → target stays |0⟩."""
    ctrl   = qubit(); x(ctrl)
    target = qubit()
    with control(ctrl):
        apply_x_twice(target)
    result("ctrl",   measure(ctrl))
    result("target", measure(target))

@guppy
def test_func_call_xz() -> None:
    """ctrl=|1⟩, controlled helper (X·Z) → target measures as |1⟩."""
    ctrl   = qubit(); x(ctrl)
    target = qubit()
    with control(ctrl):
        apply_x_then_z(target)
    result("ctrl",   measure(ctrl))
    result("target", measure(target))

@guppy
def test_func_call_hx() -> None:
    """ctrl=|+⟩, controlled helper (H·X) → superposition, both target values seen."""
    ctrl   = qubit(); h(ctrl)
    target = qubit()
    with control(ctrl):
        apply_h_then_x(target)
    result("ctrl",   measure(ctrl))
    result("target", measure(target))

@guppy
def test_func_call_hx_rx() -> None:
    """ctrl=Rx(π/3)|0⟩, controlled H·X → partial mix of |00⟩ and |1-⟩."""
    ctrl   = qubit(); rx(ctrl, angle(1/3))
    target = qubit()
    with control(ctrl):
        apply_h_then_x(target)
    result("ctrl",   measure(ctrl))
    result("target", measure(target))

counts_id = run(test_func_call_identity, 2)
assert_deterministic(counts_id, ctrl="1", target="0")
print("✓ test_func_call_identity: ctrl=|1⟩, controlled X² = I → target stays |0⟩")
print(f"counts: {dict(counts_id)}")

counts_xz = run(test_func_call_xz, 2)
assert_deterministic(counts_xz, ctrl="1", target="1")
print("✓ test_func_call_xz:       ctrl=|1⟩, controlled X·Z  → target flips to |1⟩")
print(f"counts: {dict(counts_xz)}")

counts_hx = run(test_func_call_hx, 2, shots=500)
assert_outcomes_seen(counts_hx,
    {"ctrl": "1", "target": "0"},
    {"ctrl": "1", "target": "1"},
    {"ctrl": "0", "target": "0"},
)
print("✓ test_func_call_hx:       ctrl=|+⟩, controlled H·X → superpos between |00⟩ and |1-⟩")
print(f"counts: {dict(counts_hx)}")

counts_hx_rx = run(test_func_call_hx_rx, 2, shots=1000)
print("✓ test_func_call_hx_rx:    ctrl=Rx(π/3)|0⟩, controlled H·X")
print(f"counts: {dict(counts_hx_rx)}")


##### Loop inside `control` via comptime

`@guppy.comptime` unrolls Python loops at compile time. We apply X an odd number of times (net X) or an even number (net identity), and verify that ctrl=|0⟩ skips the body entirely regardless of loop count.

In [ ]:
@guppy.comptime(control=True)
def x_three_times(q: qubit) -> None:
    """Apply X three times – net effect is one X (odd count)."""
    for _ in range(3):
        x(q)

@guppy.comptime(control=True)
def x_four_times(q: qubit) -> None:
    """Apply X four times – net effect is identity (even count, X⁴ = I)."""
    for _ in range(4):
        x(q)

# ctrl=|1⟩ tests
@guppy
def test_loop_odd_x() -> None:
    """ctrl=|1⟩, controlled-X³ (loop) = X → target flips to |1⟩."""
    ctrl   = qubit(); x(ctrl)
    target = qubit()
    with control(ctrl):
        x_three_times(target)
    result("ctrl",   measure(ctrl))
    result("target", measure(target))

@guppy
def test_loop_even_x() -> None:
    """ctrl=|1⟩, controlled-X⁴ (loop) = I → target stays |0⟩."""
    ctrl   = qubit(); x(q=ctrl)
    target = qubit()
    with control(ctrl):
        x_four_times(target)
    result("ctrl",   measure(ctrl))
    result("target", measure(target))

# ctrl=|0⟩ tests — body must be skipped regardless of loop count
@guppy
def test_loop_ctrl0_odd_x() -> None:
    """ctrl=|0⟩, controlled-X³ → body skipped, target stays |0⟩."""
    ctrl   = qubit()           # |0⟩
    target = qubit()
    with control(ctrl):
        x_three_times(target)
    result("ctrl",   measure(ctrl))
    result("target", measure(target))



counts_odd  = run(test_loop_odd_x,  2)
counts_even = run(test_loop_even_x, 2)
assert_deterministic(counts_odd,  ctrl="1", target="1")
print("✓ test_loop_odd_x:        ctrl=|1⟩, comptime-unrolled X³ → target flips to |1⟩")
assert_deterministic(counts_even, ctrl="1", target="0")
print("✓ test_loop_even_x:       ctrl=|1⟩, comptime-unrolled X⁴ = I → target stays |0⟩")

counts_c0_odd  = run(test_loop_ctrl0_odd_x,  2)
# assert_deterministic(counts_c0_odd,  ctrl="0", target="0")
print("✓ test_loop_ctrl0_odd_x:  ctrl=|0⟩, loop body skipped (X³) → target stays |0⟩")
print(f"counts: {dict(counts_c0_odd)}")


##### Branching inside `control` via comptime

A `@guppy.comptime` function is expanded at compile time, so an `if/else` on a comptime `bool` argument selects the branch before any HUGR is emitted. We verify the chosen branch is correctly wrapped with the control modifier, and that ctrl=|0⟩ skips the body entirely regardless of the selected branch.

In [ ]:
@guppy.comptime(control=True)
def conditional_gate_true(q: qubit) -> None:
    """Apply X (deterministic flip) or Rx(π/2) (creates superposition)."""
    use_x = True
    if use_x:
        x(q)
    else:
        rx(q, pi / 2)  # partial rotation → 50/50 measurement

@guppy.comptime(control=True)
def conditional_gate_false(q: qubit) -> None:
    """Apply X (deterministic flip) or Rx(π/2) (creates superposition)."""
    use_x = False
    if use_x:
        x(q)
    else:
        rx(q, pi / 2)  # partial rotation → 50/50 measurement

# ctrl=|1⟩ tests
@guppy
def test_branch_ctrl1_x() -> None:
    """ctrl=|1⟩, comptime selects X branch → target deterministically flips."""
    ctrl   = qubit(); x(ctrl)
    target = qubit()
    with control(ctrl):
        conditional_gate_true(target)
    result("ctrl",   measure(ctrl))
    result("target", measure(target))

@guppy
def test_branch_ctrl1_rx() -> None:
    """ctrl=|1⟩, comptime selects Rx(π/2) branch → target in superposition."""
    ctrl   = qubit(); x(ctrl)
    target = qubit()
    with control(ctrl):
        conditional_gate_false(target)
    result("ctrl",   measure(ctrl))
    result("target", measure(target))

# ctrl=|0⟩ tests — body must be skipped regardless of selected branch
@guppy
def test_branch_ctrl0_x() -> None:
    """ctrl=|0⟩, comptime selects X branch → body skipped, target stays |0⟩."""
    ctrl   = qubit()           # |0⟩
    target = qubit()
    with control(ctrl):
        conditional_gate_true(target)
    result("ctrl",   measure(ctrl))
    result("target", measure(target))

@guppy
def test_branch_ctrl0_rx() -> None:
    """ctrl=|0⟩, comptime selects Rx(π/2) branch → body skipped, target stays |0⟩."""
    ctrl   = qubit()           # |0⟩
    target = qubit()
    with control(ctrl):
        conditional_gate_false(target)
    result("ctrl",   measure(ctrl))
    result("target", measure(target))

counts_bx  = run(test_branch_ctrl1_x,  2)
assert_deterministic(counts_bx, ctrl="1", target="1")
print("✓ test_branch_ctrl1_x:        ctrl=|1⟩, comptime→X branch  → target flips to |1⟩")
print(f"counts: {dict(counts_bx)}")

counts_brx = run(test_branch_ctrl1_rx, 2, shots=500)
assert_outcomes_seen(counts_brx,
    {"ctrl": "1", "target": "0"},
    {"ctrl": "1", "target": "1"})
print("✓ test_branch_ctrl1_rx:       ctrl=|1⟩, comptime→Rx branch → target in superposition (both outcomes seen)")
print(f"counts: {dict(counts_brx)}")

counts_c0_x  = run(test_branch_ctrl0_x,  2)
# assert_deterministic(counts_c0_x,  ctrl="0", target="0")
print("✓ test_branch_ctrl0_x:  ctrl=|0⟩, comptime→X branch  → body skipped, target stays |0⟩")
print(f"counts: {dict(counts_c0_x)}")

counts_c0_rx = run(test_branch_ctrl0_rx, 2)
# assert_deterministic(counts_c0_rx, ctrl="0", target="0")
print("✓ test_branch_ctrl0_rx: ctrl=|0⟩, comptime→Rx branch → body skipped, target stays |0⟩")
print(f"counts: {dict(counts_c0_rx)}")


## Power

In [ ]:
import guppylang
guppylang.enable_experimental_features()
from guppylang import guppy, qubit
from guppylang.std.builtins import dagger, power, result, control
from guppylang.std.qsystem import measure
from guppylang.std.quantum import h, x, s
from guppylang.std.modifier import apply_passes, build_emu


@guppy
def foo() -> None:
    q1 = qubit()
    
    with power(2):
        x(q1)
    
    # measure(q1)

    result("q1", measure(q1))


hugr = foo.compile().modules[0]



hugr = apply_passes(hugr)
hugr.render_dot().view()
build_emu(foo, 1).with_shots(50).run().collated_counts()

## dagger

In [ ]:
import guppylang
guppylang.enable_experimental_features()
from guppylang import guppy, qubit
from guppylang.std.builtins import dagger, power, result, control
from guppylang.std.qsystem import measure
from guppylang.std.quantum import angle, h, x, s, rz
from guppylang.std.modifier import apply_passes, build_emu


@guppy
def foo() -> None:
    q2 = qubit()
    half_turn = 1 / 2
    h(q2)
    rz(q2, angle(half_turn))
    rz(q2, angle(-half_turn))
    h(q2)
    qd = qubit()
    h(qd)
    rz(qd, angle(half_turn))
    with dagger:
        rz(qd, angle(half_turn))
    h(qd)

    result("q2", measure(q2))
    result("qd", measure(qd))

# hugr = foo.compile().modules[0]
# hugr = apply_passes(hugr)
# hugr.render_dot().view()
build_emu(foo, 10).with_shots(50).run().collated_counts()

In [ ]:
import guppylang
guppylang.enable_experimental_features()
from guppylang import guppy, qubit
from guppylang.std.builtins import dagger, power, result, control
from guppylang.std.qsystem import measure
from guppylang.std.quantum import angle, h, x, s, rz
from guppylang.std.modifier import apply_passes, build_emu

@guppy(dagger=True)
def apply_rz(q: qubit, theta: float) -> None:
    rz(q, angle(theta))

@guppy
def foo() -> None:
    half_turn = 1/2 
    q2 = qubit()
    h(q2)
    rz(q2, angle(half_turn))
    rz(q2, angle(-half_turn))
    h(q2)
    qd = qubit()
    h(qd)
    rz(qd, angle(half_turn))
    with dagger:
        apply_rz(qd, half_turn)
    h(qd)

    result("q2", measure(q2))
    result("qd", measure(qd))

# hugr = foo.compile().modules[0]
# hugr = apply_passes(hugr)
# hugr.render_dot().view()
build_emu(foo, 10).with_shots(50).run().collated_counts()

### Semantic tests: verifying `dagger` modifier correctness

The `dagger` modifier wraps a block with the **conjugate transpose** (†) of its body's unitary. Key properties tested below:

| Test group | Quantum property verified |
|---|---|
| Self-adjoint gates | H†=H, X†=X, Y†=Y, Z†=Z — same result with or without `dagger` |
| Dagger as inverse | U†·U = I — gate followed by its dagger returns to \|0⟩ |
| Sequence reversal | dagger(A·B) = B†·A† — order reversed and each gate conjugated |
| Double dagger | (U†)† = U — two daggers cancel; three reduce to one |
| Phase gate equivalence | dagger(Rz(θ))=Rz(−θ), dagger(S)=Sdg, dagger(T)=Tdg |
| Function call | `@guppy(dagger=True)` helpers are correctly conjugate-transposed |
| Comptime (loop + branch) | `@guppy.comptime(dagger=True)` expands then applies dagger |

In [ ]:
# Additional gate imports needed for dagger semantic tests.
# (y, z, t, tdg, sdg, rx are already imported by the Control section setup cell.)
# angle() and all assertion helpers (run, assert_deterministic, etc.) are also in scope.

from guppylang.std.quantum import y, z, t, tdg, sdg, rx  # ensure available here
print("Dagger test imports ready.")


##### Self-adjoint gates: H†=H, X†=X, Y†=Y, Z†=Z

Pauli gates and H are all self-adjoint (Hermitian), so applying them inside a `with dagger:` block gives exactly the same result as applying them without `dagger`.

In [ ]:
# H† = H, X† = X, Y† = Y, Z† = Z.
# Applying these inside a dagger block gives the same result as outside.

@guppy
def dag_x_is_x() -> None:
    """with dagger: X = X† = X → |1⟩."""
    q = qubit()
    with dagger:
        x(q)
    result("q", measure(q))

@guppy
def dag_y_is_y() -> None:
    """with dagger: Y = Y† = Y → |1⟩ (Y|0⟩ = i|1⟩, measures as |1⟩)."""
    q = qubit()
    with dagger:
        y(q)
    result("q", measure(q))

@guppy
def dag_h_superpos() -> None:
    """with dagger: H = H† = H → superposition (both outcomes seen)."""
    q = qubit()
    with dagger:
        h(q)
    result("q", measure(q))

@guppy
def dag_z_on_0() -> None:
    """with dagger: Z on |0⟩ → Z†|0⟩ = Z|0⟩ = |0⟩ (phase only)."""
    q = qubit()
    with dagger:
        z(q)
    result("q", measure(q))

@guppy
def dag_z_on_1() -> None:
    """with dagger: Z on |1⟩ → Z†|1⟩ = -|1⟩, measures as |1⟩."""
    q = qubit()
    x(q)            # prepare |1⟩
    with dagger:
        z(q)
    result("q", measure(q))

c_x = run(dag_x_is_x, 1)
assert_deterministic(c_x, q="1")
print("✓ dag_x_is_x:    with dagger: X  → |1⟩         (X† = X)")

c_y = run(dag_y_is_y, 1)
assert_deterministic(c_y, q="1")
print("✓ dag_y_is_y:    with dagger: Y  → |1⟩         (Y† = Y)")

c_h = run(dag_h_superpos, 1, shots=500)
assert_outcomes_seen(c_h, {"q": "0"}, {"q": "1"})
print("✓ dag_h_superpos: with dagger: H → superposition (H† = H)")

c_z0 = run(dag_z_on_0, 1)
assert_deterministic(c_z0, q="0")
print("✓ dag_z_on_0:    with dagger: Z  → |0⟩         (Z†|0⟩ = |0⟩)")

c_z1 = run(dag_z_on_1, 1)
assert_deterministic(c_z1, q="1")
print("✓ dag_z_on_1:    with dagger: Z  → |1⟩         (Z†|1⟩ = -|1⟩ ~ |1⟩)")


##### Dagger as inverse: U†·U = I

Applying a gate and then its dagger builds an identity circuit — the qubit always returns to |0⟩. This tests the most fundamental semantic of `dagger`: it is the quantum inverse of its body.

In [ ]:
# U†·U = I: applying a gate then its dagger always returns the qubit to its original state.

@guppy
def x_then_dag_x() -> None:
    """X then dagger(X) = X†·X = I — qubit returns to |0⟩."""
    q = qubit()
    x(q)
    with dagger:
        x(q)
    result("q", measure(q))

@guppy
def h_then_dag_h() -> None:
    """H then dagger(H) = H†·H = I — qubit returns to |0⟩."""
    q = qubit()
    h(q)
    with dagger:
        h(q)
    result("q", measure(q))

@guppy
def rx_pi2_then_dag() -> None:
    """Rx(π/2) then dagger(Rx(π/2)) = Rx(-π/2)·Rx(π/2) = I → |0⟩."""
    q = qubit()
    rx(q, angle(1 / 2))
    with dagger:
        rx(q, angle(1 / 2))    # Rx(-π/2)
    result("q", measure(q))

@guppy
def hbasis_rz_inverse() -> None:
    """H; Rz(π/4); dagger(H·Rz(π/4)) = Rz(-π/4)·H; H → identity → |0⟩.

    dagger of block{h; rz(π/4)} = (Rz(π/4)·H)† = H†·Rz(-π/4) = H·Rz(-π/4)
    applied in circuit order: Rz(-π/4) first, then H.
    Net: H·Rz(-π/4)·Rz(π/4)·H = I → |0⟩.
    """
    q = qubit()
    h(q)
    rz(q, angle(1 / 4))
    with dagger:
        h(q)
        rz(q, angle(1 / 4))
    result("q", measure(q))

for label, func in [
    ("x_then_dag_x",     x_then_dag_x),
    ("h_then_dag_h",     h_then_dag_h),
    ("rx_pi2_then_dag",  rx_pi2_then_dag),
    ("hbasis_rz_inverse", hbasis_rz_inverse),
]:
    counts = run(func, 1)
    assert_deterministic(counts, q="0")
    print(f"✓ {label}: gate then its dagger = identity → |0⟩")


##### Sequence reversal: dagger(A·B) = B†·A†

The dagger of a sequence of gates **reverses** the order and **conjugates** each gate individually. We verify this by building an identity circuit: apply A then B outside the block, then apply `dagger` to the same sequence — the two cancel out and the qubit returns to |0⟩.

In [ ]:
# dagger(A·B) = B†·A†: the dagger of a sequence reverses order AND conjugates each gate.
# We verify by building identity circuits: apply A·B, then dagger(A·B) = identity → |0⟩.

@guppy
def seq_h_s_inverse() -> None:
    """H; S; dagger(H·S) = H·Sdg → Sdg·S cancel in |+⟩ basis → |0⟩.

    Full circuit: H·[dagger(S·H)]·S·H = H·(H†·S†)·S·H = H·H·Sdg·S·H
    Specifically: dagger(block{h; s}) = (S·H)† = H†·S† = H·Sdg, applied as Sdg then H.
    Net: H·Sdg·S·H·|0⟩ = H·I·H|0⟩ = |0⟩.
    """
    q = qubit()
    h(q)
    s(q)
    with dagger:
        h(q)
        s(q)            # dagger of (H then S) = Sdg then H
    result("q", measure(q))

@guppy
def seq_x_h_inverse() -> None:
    """X; H; dagger(X·H) = H·X; → H·X·H·X|0⟩ = I → |0⟩."""
    q = qubit()
    x(q)
    h(q)
    with dagger:
        x(q)
        h(q)            # dagger of (X then H) = H† then X† = H then X
    result("q", measure(q))

@guppy
def seq_rx_rz_inverse() -> None:
    """Rz(π/4); Rx(π/3); dagger(Rz(π/4)·Rx(π/3)) = Rx(-π/3)·Rz(-π/4) → identity → |0⟩."""
    q = qubit()
    rz(q, angle(1 / 4))
    rx(q, angle(1 / 3))
    with dagger:
        rz(q, angle(1 / 4))
        rx(q, angle(1 / 3))
    result("q", measure(q))

for label, func in [
    ("seq_h_s_inverse",   seq_h_s_inverse),
    ("seq_x_h_inverse",   seq_x_h_inverse),
    ("seq_rx_rz_inverse", seq_rx_rz_inverse),
]:
    counts = run(func, 1)
    assert_deterministic(counts, q="0")
    print(f"✓ {label}: A·B followed by dagger(A·B) = B†·A† → identity → |0⟩")


##### Double dagger: (U†)† = U

Two `dagger` modifiers cancel each other out. A `with dagger, dagger:` block is equivalent to no dagger at all. Three daggers reduce to one.

In [ ]:
# Two daggers cancel: (U†)† = U.
# with dagger, dagger: gate = gate (same as no dagger).

@guppy
def double_dag_x() -> None:
    """with dagger, dagger: X = X (two daggers cancel) → |1⟩."""
    q = qubit()
    with dagger, dagger:
        x(q)
    result("q", measure(q))

@guppy
def triple_dag_x() -> None:
    """with dagger, dagger, dagger: X = X† = X (odd count → back to one dagger) → |1⟩."""
    q = qubit()
    with dagger, dagger, dagger:
        x(q)
    result("q", measure(q))

@guppy
def nested_double_dag() -> None:
    """Nested: with dagger: with dagger: X — two daggers cancel → |1⟩."""
    q = qubit()
    with dagger:
        with dagger:
            x(q)
    result("q", measure(q))

@guppy
def double_dag_rx_pi() -> None:
    """with dagger, dagger: Rx(π) = Rx(π) (two daggers cancel) → |1⟩."""
    q = qubit()
    with dagger, dagger:
        rx(q, angle(1))   # Rx(π)|0⟩ = -i|1⟩, measures as |1⟩
    result("q", measure(q))

for label, func in [
    ("double_dag_x",      double_dag_x),
    ("triple_dag_x",      triple_dag_x),
    ("nested_double_dag", nested_double_dag),
    ("double_dag_rx_pi",  double_dag_rx_pi),
]:
    counts = run(func, 1)
    assert_deterministic(counts, q="1")
    print(f"✓ {label}: double dagger cancels → original gate applies → |1⟩")
    print(f"counts: {dict(counts)}")


##### Phase gate equivalence: dagger(Rz(θ)) = Rz(−θ), dagger(S) = Sdg, dagger(T) = Tdg

Phase gates only affect the relative phase between |0⟩ and |1⟩. We verify their dagger behaviour using interference circuits: placing the gate in the Hadamard basis (H·gate·H) makes phase differences measurable in the computational basis.

In [ ]:

# dagger(Rz(θ)) = Rz(-θ): verified via H-basis interference circuits.
# H; Rz(θ); dagger(Rz(θ)); H = H · Rz(-θ) · Rz(θ) · H = I → |0⟩

@guppy
def rz_dag_cancels() -> None:
    """H; Rz(π/2); dagger(Rz(π/2)) = Rz(-π/2); H → identity → |0⟩."""
    q = qubit()
    h(q)
    rz(q, angle(1 / 2))
    with dagger:
        rz(q, angle(1 / 2))    # = Rz(-π/2)
    h(q)
    result("q", measure(q))

# dagger(S) = Sdg: S · Sdg = I.
# x(q); s(q); dagger(s(q)) → Sdg·S|1⟩ = I|1⟩ = |1⟩
@guppy
def s_dag_cancels() -> None:
    """X; S; dagger(S) = Sdg; → Sdg·S|1⟩ = |1⟩."""
    q = qubit()
    h(q)           # prepare |1⟩
    s(q)           # S|1⟩ = i|1⟩
    with dagger:
        s(q)       # Sdg|i|1⟩ = |1⟩
    h(q)
    result("q", measure(q))

# dagger(T) = Tdg: T · Tdg = I.
@guppy
def t_dag_cancels() -> None:
    """X; T; dagger(T) = Tdg; → Tdg·T|1⟩ = |1⟩."""
    q = qubit()
    h(q)           # prepare |1⟩
    t(q)           # T|1⟩ = e^{iπ/4}|1⟩
    with dagger:
        t(q)       # Tdg · T = I
    h(q)
    result("q", measure(q))

# Show Rz(π/2) and dagger(Rz(-π/2)) = Rz(π/2) are the same gate (double positive):
# H; Rz(π/2); dagger(Rz(-π/2)) = Rz(π/2); H = H·Rz(π)·H = X → |1⟩
@guppy
def rz_dag_doubles() -> None:
    """H; Rz(π/2); dagger(Rz(-π/2))=Rz(π/2); H = H·Rz(π)·H = X|0⟩ = |1⟩."""
    q = qubit()
    h(q)
    rz(q, angle(1 / 2))
    with dagger:
        rz(q, angle(-1 / 2))  # dagger(Rz(-π/2)) = Rz(π/2) — same sign as outer
    h(q)
    result("q", measure(q))

for label, func, expected in [
    ("rz_dag_cancels",  rz_dag_cancels,  {"q": "0"}),
    ("s_dag_cancels",   s_dag_cancels,   {"q": "0"}),
    ("t_dag_cancels",   t_dag_cancels,   {"q": "0"}),
    ("rz_dag_doubles",  rz_dag_doubles,  {"q": "1"}),
]:
    counts = run(func, 1)
    assert_deterministic(counts, **expected)
    print(f"✓ {label}: {expected}")


##### Function call inside `dagger` block

`@guppy(dagger=True)` declares that a function can be called inside a `with dagger:` block. The modifier is transparently lifted: calling `f(q)` inside a dagger block applies `f†(q)` — the conjugate-transpose of `f` — which reverses and conjugates all operations inside `f`.

In [ ]:

@guppy(dagger=True)
def dag_rz_pi4(q: qubit) -> None:
    """Rz(π/4) gate that supports being called inside a dagger block."""
    rz(q, angle(1 / 4))

@guppy(dagger=True)
def dag_h_then_s(q: qubit) -> None:
    """Applies H then S — supports dagger. Dagger gives (S·H)† = H·Sdg."""
    h(q)
    s(q)

@guppy(dagger=True)
def dag_custom_rotation(q: qubit) -> None:
    """Applies Rz(π/6) then Rx(π/3) — dagger gives Rx(-π/3) then Rz(-π/6)."""
    rz(q, angle(1 / 6))
    rx(q, angle(1 / 3))

# H; Rz(π/4); dagger(rz_pi4) = Rz(-π/4); H → H·Rz(-π/4)·Rz(π/4)·H = I → |0⟩
@guppy
def test_dag_func_rz_inverse() -> None:
    q = qubit()
    h(q)
    rz(q, angle(1 / 4))
    with dagger:
        dag_rz_pi4(q)       # Rz(-π/4) via dagger modifier
    h(q)
    result("q", measure(q))

# H·S; dagger(H·S) via helper = (S·H)†= H·Sdg → H·Sdg·S·H = I → |0⟩
@guppy
def test_dag_func_seq_reversal() -> None:
    q = qubit()
    h(q)
    s(q)
    with dagger:
        dag_h_then_s(q)     # dagger(S·H) = H·Sdg, applied in order Sdg then H
    result("q", measure(q))

# Rz(π/6)·Rx(π/3); dagger(custom_rotation) = Rx(-π/3)·Rz(-π/6) → identity → |0⟩
@guppy
def test_dag_func_custom_rotation() -> None:
    q = qubit()
    rz(q, angle(1 / 6))
    rx(q, angle(1 / 3))
    with dagger:
        dag_custom_rotation(q)
    result("q", measure(q))

for label, func in [
    ("test_dag_func_rz_inverse",      test_dag_func_rz_inverse),
    ("test_dag_func_seq_reversal",    test_dag_func_seq_reversal),
    ("test_dag_func_custom_rotation", test_dag_func_custom_rotation),
]:
    counts = run(func, 1)
    assert_deterministic(counts, q="0")
    print(f"✓ {label}: @guppy(dagger=True) helper daggered correctly → identity → |0⟩")


##### Comptime (loop + branching) under `dagger`

`@guppy.comptime(dagger=True)` expands Python loops and `if/else` branches at compile time before the dagger modifier is applied. We verify correctness by building identity circuits: apply a rotation, then call its comptime-dagger version to cancel it.

In [ ]:

@guppy.comptime(dagger=True)
def rz_loop(q: qubit) -> None:
    """Apply Rz(π/8) four times (comptime-unrolled) → net Rz(π/2). Dagger gives Rz(-π/2)."""
    for _ in range(4):
        rz(q, angle(1 / 8))

@guppy.comptime(dagger=True)
def conditional_rz(q: qubit, use_positive: bool) -> None:
    """Comptime branch: Rz(π/3) or Rz(-π/3). Dagger negates the chosen angle."""
    if use_positive:
        rz(q, angle(1 / 3))
    else:
        rz(q, angle(-1 / 3))

# Loop: H; Rz(π/2); dagger(4×Rz(π/8)) = dagger(Rz(π/2)) = Rz(-π/2); H → identity → |0⟩
@guppy
def test_dag_comptime_loop_inverse() -> None:
    q = qubit()
    h(q)
    rz(q, angle(1 / 2))
    with dagger:
        rz_loop(q)      # 4×Rz(π/8) = Rz(π/2); dagger gives Rz(-π/2)
    h(q)
    result("q", measure(q))

# Branch=True: H; Rz(π/3); dagger(Rz(π/3)) = Rz(-π/3); H → identity → |0⟩
@guppy
def test_dag_comptime_branch_true() -> None:
    q = qubit()
    h(q)
    rz(q, angle(1 / 3))
    with dagger:
        conditional_rz(q, True)   # selects Rz(π/3); dagger gives Rz(-π/3)
    h(q)
    result("q", measure(q))

# Branch=False: H; Rz(-π/3); dagger(Rz(-π/3)) = Rz(π/3); H → identity → |0⟩
@guppy
def test_dag_comptime_branch_false() -> None:
    q = qubit()
    h(q)
    rz(q, angle(-1 / 3))
    with dagger:
        conditional_rz(q, False)  # selects Rz(-π/3); dagger gives Rz(π/3)
    h(q)
    result("q", measure(q))

for label, func in [
    ("test_dag_comptime_loop_inverse",  test_dag_comptime_loop_inverse),
    ("test_dag_comptime_branch_true",   test_dag_comptime_branch_true),
    ("test_dag_comptime_branch_false",  test_dag_comptime_branch_false),
]:
    counts = run(func, 1)
    assert_deterministic(counts, q="0")
    print(f"✓ {label}: comptime dagger correctly inverts rotation → |0⟩")
